In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

sys.path.insert(0, '..')  # dataset.py lives one level up, shared across sector/industry notebooks
import dataset
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import model as model_module
import numpy as np
import pandas as pd
import torch
import train as train_module
from sklearn.metrics import confusion_matrix

from investalyze.analysis import encodings
from investalyze.ingest import storage

plt.rcParams['figure.dpi'] = 130


In [ ]:
TICKERS = None  # list[str] of specific tickers, int for a random sample of that many, or None for all
UNIVERSE = None  # name of a universe saved by the screener app (data/universes/<name>.csv); overrides TICKERS when set
EXCLUDE_TICKERS = []  # tickers to always leave out, regardless of TICKERS or UNIVERSE

LABEL_COLUMN = 'Industry'  # 'Sector' or 'Industry', from the companies table

SEED = 0  # used when TICKERS is an int, and when VALID_METHOD is 'random'
WINDOW_LENGTH = 30
STRIDE = 15

VALID_FRAC = 0.3
VALID_METHOD = 'random'  # 'recent' = time-based (no leakage) / 'random' = random per-window
TEST_N = 1  # always recent: the last N windows of each ticker

ENCODER = encodings.zscore  # swap by hand: RebaseTo100 / RebaseTo1 / encodings.zscore / encodings.minmax
BATCH_SIZE = 1024 * 24
EPOCHS = 300000000000000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

HIDDEN = 64
DROPOUT = 0.3
N_BLOCKS = 10  # residual conv blocks (skip connection per block)
KERNEL_START = 3  # kernel size of the stem conv + first residual blocks
KERNEL_END = 9  # kernel size reached by the last residual block (quadratic ramp, odd sizes only)

LR = 1e-3
WEIGHT_DECAY = 0.01  # AdamW L2-ish regularization, applied directly to weights not through the gradient
LR_FACTOR = 0.5  # ReduceLROnPlateau: multiply lr by this on plateau
LR_PATIENCE = 20  # ReduceLROnPlateau: epochs of no valid_loss improvement before reducing lr
LR_METRIC = 'valid'  # 'valid' normally / 'train' when deliberately overfitting to sanity-check capacity

EARLY_STOP_PATIENCE = 100000000  # stop if valid_loss hasn't improved in this many epochs
EVAL_CHECKPOINT = 'valid_loss'  # 'last' / 'train_loss' / 'valid_loss' / 'train_acc' / 'valid_acc'


In [ ]:
DATA_ROOT = Path('../../data')

con = storage.connect(DATA_ROOT, read_only=True)
if UNIVERSE is not None:
    tickers = [t for t in dataset.load_universe(UNIVERSE, DATA_ROOT) if t not in EXCLUDE_TICKERS]
elif isinstance(TICKERS, int):
    tickers = dataset.sample_tickers(con, TICKERS, seed=SEED, exclude=EXCLUDE_TICKERS)
else:
    tickers = TICKERS
series = dataset.get_ohlcv_series(con, tickers, exclude=EXCLUDE_TICKERS)
ticker_labels = dataset.get_ticker_labels(con, LABEL_COLUMN)
con.close()
print('tickers used', tickers if tickers is not None else 'ALL')

channels, meta = dataset.build_windows(series, window_length=WINDOW_LENGTH, stride=STRIDE)
channels, meta = dataset.attach_labels(channels, meta, ticker_labels)
n_tickers, n_labels = meta['Ticker'].nunique(), meta['label'].nunique()
print(f"windows {meta.shape[0]} across {n_tickers} tickers and {n_labels} industrys")


In [ ]:
train_mask, valid_mask, test_mask = dataset.split_windows(
    meta, valid_frac=VALID_FRAC, valid_method=VALID_METHOD, test_n=TEST_N, seed=SEED
)

labels, class_names = dataset.encode_labels(meta, train_mask, label_col='label')
n_classes = len(class_names)

print('train windows', int(train_mask.sum()), 'valid windows', int(valid_mask.sum()), 'test windows', int(test_mask.sum()))
pd.Series(labels[train_mask]).value_counts().sort_index()


In [ ]:
X = dataset.encode_windows(channels, ENCODER)
X_train, y_train = X[train_mask], labels[train_mask]
X_valid, y_valid = X[valid_mask], labels[valid_mask]
X_test, y_test = X[test_mask], labels[test_mask]

weights = dataset.class_weights(y_train, n_classes)
train_loader = train_module.make_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = train_module.make_loader(X_valid, y_valid, batch_size=BATCH_SIZE, shuffle=False)
test_loader = train_module.make_loader(X_test, y_test, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
net = model_module.OHLCVClassifierCNN(
    n_channels=5,
    n_classes=n_classes,
    hidden=HIDDEN,
    dropout=DROPOUT,
    n_blocks=N_BLOCKS,
    kernel_start=KERNEL_START,
    kernel_end=KERNEL_END,
).to(DEVICE)
optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)
history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
best = train_module.new_best_tracker()


In [ ]:
net


In [ ]:
CHANGE_LR = False  # set True (and NEW_LR below), then run this cell to change lr mid-training
NEW_LR = 0.1

if CHANGE_LR:
    for g in optimizer.param_groups:
        g['lr'] = NEW_LR
print(f'lr = {optimizer.param_groups[0]["lr"]:.2e}')


In [ ]:
print(f'Baseline loss: {np.log(n_classes):.4f} | acc: {1 / n_classes:.4f}')

# re-run this cell (unchanged) to keep training the same net/optimizer for EPOCHS more
history, best = train_module.train_model(
    net,
    train_loader,
    valid_loader,
    optimizer,
    epochs=EPOCHS,
    class_weights=weights,
    n_classes=n_classes,
    device=DEVICE,
    history=history,
    scheduler=scheduler,
    scheduler_metric=LR_METRIC,
    best=best,
    early_stop_patience=EARLY_STOP_PATIENCE,
)


In [ ]:
for metric_name, info in best.items():
    print(f'best {metric_name}: epoch {info["epoch"] + 1}  value={info["value"]:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss'][10:], label='train')
axes[0].plot(history['valid_loss'][10:], label='valid')
axes[0].set_title('loss')
axes[0].legend()
axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['valid_acc'], label='valid')
axes[1].axhline(1 / n_classes, color='gray', linestyle='--', label='random baseline')
axes[1].set_title('macro-accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
if EVAL_CHECKPOINT != 'last':
    ckpt = best[EVAL_CHECKPOINT]
    net.load_state_dict(ckpt['state'])
    print(f'loaded {EVAL_CHECKPOINT} checkpoint from epoch {ckpt["epoch"] + 1}  value={ckpt["value"]:.4f}')
else:
    print('using last-epoch weights')


In [ ]:
valid_preds, valid_true = train_module.predict(net, valid_loader, DEVICE)
cm_valid = confusion_matrix(valid_true, valid_preds, labels=range(n_classes))
row_sums = cm_valid.sum(axis=1, keepdims=True)
cm_valid_norm = np.divide(cm_valid, row_sums, out=np.zeros_like(cm_valid, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm_valid_norm, cmap='viridis', norm=mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=1), interpolation='nearest')
if n_classes <= 40:
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(class_names)
else:
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('valid set (row-normalized, gamma=0.4)')
plt.colorbar(im, label='fraction of true class')
plt.tight_layout()
plt.show()


In [ ]:
cm_valid_offdiag = cm_valid.copy()
np.fill_diagonal(cm_valid_offdiag, 0)
i_idx, j_idx = np.nonzero(cm_valid_offdiag)
counts = cm_valid_offdiag[i_idx, j_idx]
order = np.argsort(-counts)[:15]
valid_pairs = [(class_names[i], class_names[j], int(c)) for i, j, c in zip(i_idx[order], j_idx[order], counts[order])]
valid_pairs


In [ ]:
preds, y_true = train_module.predict(net, test_loader, DEVICE)
test_acc = (preds == y_true).mean()
test_macro_acc = train_module.macro_accuracy(torch.tensor(preds), torch.tensor(y_true), n_classes)
print(f'test accuracy={test_acc:.3f}  macro-accuracy={test_macro_acc:.3f}')
cm = confusion_matrix(y_true, preds, labels=range(n_classes))
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm_norm, cmap='viridis', norm=mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=1), interpolation='nearest')
if n_classes <= 40:
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(class_names)
else:
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('test set (row-normalized, gamma=0.4)')
plt.colorbar(im, label='fraction of true class')
plt.tight_layout()
plt.show()


In [ ]:
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)
i_idx, j_idx = np.nonzero(cm_offdiag)
counts = cm_offdiag[i_idx, j_idx]
order = np.argsort(-counts)[:150]
pairs = [(class_names[i], class_names[j], int(c)) for i, j, c in zip(i_idx[order], j_idx[order], counts[order])]
pairs


In [ ]:
def worst_predicted(cm_offdiag, top=20):
    fp_counts = cm_offdiag.sum(axis=0)  # column sums: how often each class is wrongly predicted
    n_sources = (cm_offdiag > 0).sum(axis=0)  # distinct true classes wrongly predicted as this one
    order = np.argsort(-fp_counts)[:top]
    return order, fp_counts, n_sources


for name, offdiag in [('valid', cm_valid_offdiag), ('test', cm_offdiag)]:
    order, fp_counts, n_sources = worst_predicted(offdiag)
    print(f'{name}:')
    for idx in order:
        pct_sources = n_sources[idx] / n_classes * 100
        print(
            f'  {class_names[idx]:<28}  wrongly predicted {int(fp_counts[idx])} times, '
            f'from {int(n_sources[idx])} different classes ({pct_sources:.1f}% of all classes)'
        )
    print()
